In [41]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from copy import deepcopy
import random
from collections import deque
import time
import json
import os
from datetime import datetime

# ==============================================================
# CONFIGURATION
# ==============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

NUM_NODES = 10

# IMPORTANT: State dimension for routing is ONE-HOT encoding (NUM_NODES)
ROUTING_STATE_DIM = NUM_NODES  # This is 10, not 5!
ACTION_DIM = NUM_NODES

EDGE_DELAY_LOW = 3e-3
EDGE_DELAY_HIGH = 5e-3
CAPACITY_LOW = 100
CAPACITY_HIGH = 300

# ==============================================================
# GRAPH ENVIRONMENT
# ==============================================================

class TETRISGraphEnv:
    def __init__(self, delays=None):
        self.num_nodes = NUM_NODES
        if delays is None:
            self.delays = np.random.uniform(
                EDGE_DELAY_LOW, EDGE_DELAY_HIGH,
                (NUM_NODES, NUM_NODES)
            ).astype(np.float32)
        else:
            self.delays = delays.astype(np.float32)
        
        self.capacity = np.random.uniform(
            CAPACITY_LOW, CAPACITY_HIGH,
            (NUM_NODES, NUM_NODES)
        ).astype(np.float32)
        self.utilization = np.zeros((NUM_NODES, NUM_NODES), dtype=np.float32)

    def reset_task(self):
        self.src = np.random.randint(0, self.num_nodes)
        self.dst = np.random.randint(0, self.num_nodes)
        while self.dst == self.src:
            self.dst = np.random.randint(0, self.num_nodes)
        self.current = self.src
        self.total_delay = 0.0
        self.hops = 0
        self.deadline = np.random.uniform(3e-3, 5e-3)
        return self._get_state()

    def _get_state(self):
        # One-hot encoding of current node (dimension = NUM_NODES)
        s = np.zeros(self.num_nodes, dtype=np.float32)
        s[self.current] = 1.0
        return s

    def step(self, action):
        if action == self.current:
            return self._get_state(), -1.0, False, {}
        
        delay = self.delays[self.current, action]
        self.total_delay += delay
        self.hops += 1
        self.current = action
        
        done = self.current == self.dst
        
        if done:
            reward = 10.0 - (self.total_delay / self.deadline) * 5.0
        else:
            reward = -delay * 1000.0
        
        if self.total_delay > self.deadline:
            reward -= 5.0
            done = True
        
        if self.hops > self.num_nodes * 2:
            reward -= 10.0
            done = True
            
        return self._get_state(), reward, done, {}

# ==============================================================
# ROUTING DQN (Corrected dimensions)
# ==============================================================

class RoutingDQN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(ROUTING_STATE_DIM, 128),  # 10 -> 128
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, ACTION_DIM)  # 128 -> 10
        )
    
    def forward(self, x):
        return self.net(x)

# ==============================================================
# PATH ESTIMATOR
# ==============================================================

class PathEstimator:
    def __init__(self):
        self.env = TETRISGraphEnv()
        self.model = RoutingDQN().to(DEVICE)
        self.model.eval()
    
    def estimate_path_delay(self, n_samples=1):
        delays, hops = [], []
        max_steps = 50
        
        for _ in range(n_samples):
            state = self.env.reset_task()
            done = False
            steps = 0
            
            while not done and steps < max_steps:
                s = torch.from_numpy(state).float().unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    if random.random() < 0.1:
                        action = random.randint(0, NUM_NODES - 1)
                    else:
                        action = self.model(s).argmax(dim=1).item()
                
                state, _, done, _ = self.env.step(action)
                steps += 1
            
            delays.append(self.env.total_delay)
            hops.append(self.env.hops)
        
        return float(np.mean(delays)), float(np.mean(hops))

# ==============================================================
# FOG QUEUE
# ==============================================================

class FogQueueFCFS:
    def __init__(self):
        self.queue = []
    
    def add_task(self, service_time):
        self.queue.append(service_time)
    
    def step(self, dt):
        self.queue = [t - dt for t in self.queue if t - dt > 0]
    
    def backlog(self):
        return float(sum(self.queue))

# ==============================================================
# OFFLOADING ENVIRONMENT
# ==============================================================

class TETRISOffloadEnv:
    def __init__(self, path_estimator, num_tasks=200):
        self.path_estimator = path_estimator
        self.num_tasks = num_tasks
        self.dt = 1e-3
        self.drop_deadline = 10e-3
        self.reset()
    
    def reset(self):
        self.fog = FogQueueFCFS()
        self.task_id = 0
        self.total_delay = 0.0
        self.total_hops = 0.0
        self.dropped = 0
        self.processed = 0
        self.local_processed = 0
        self.offload_processed = 0
        self.last_features = np.zeros(3, dtype=np.float32)
        self.cached_delay = 0.0
        self.cached_hops = 0.0
        self.cache_counter = 0
        return self._get_obs()
    
    def _get_obs(self):
        return np.concatenate([
            self.last_features,
            np.array([self.fog.backlog()], dtype=np.float32)
        ], dtype=np.float32)
    
    def step(self, action):
        if self.task_id >= self.num_tasks:
            return self._get_obs(), 0.0, True, {}
        
        local_delay = np.random.uniform(6e-3, 8e-3)
        
        if self.cache_counter == 0:
            self.cached_delay, self.cached_hops = self.path_estimator.estimate_path_delay(n_samples=1)
        self.cache_counter = (self.cache_counter + 1) % 10
        
        path_delay = self.cached_delay
        hops = self.cached_hops
        fog_delay = max(0.0, np.random.normal(0.0, 1e-3))
        
        self.last_features = np.array([local_delay, path_delay, fog_delay], dtype=np.float32)
        
        if action == 0:
            latency = local_delay
            energy = np.random.uniform(2.0, 3.0)
            self.local_processed += 1
        else:
            service = path_delay + fog_delay
            latency = self.fog.backlog() + service
            self.fog.add_task(service)
            energy = np.random.uniform(0.5, 1.5)
            self.offload_processed += 1
        
        self.fog.step(self.dt)
        
        self.total_delay += latency
        self.total_hops += hops
        self.processed += 1
        self.task_id += 1
        
        reward = -latency * 100
        
        if latency > self.drop_deadline:
            self.dropped += 1
            reward -= 50.0
        
        done = self.task_id >= self.num_tasks
        return self._get_obs(), reward, done, {}
    
    def metrics(self):
        processed = max(1, self.processed)
        return {
            "avg_delay": self.total_delay / processed,
            "hit_ratio": 1.0 - self.dropped / processed,
            "avg_hops": self.total_hops / processed,
            "deadline_violation": self.dropped / processed,
            "local_ratio": self.local_processed / processed,
            "offload_ratio": self.offload_processed / processed
        }

# ==============================================================
# OFFLOADING DQN
# ==============================================================

class OffloadDQN(nn.Module):
    def __init__(self, state_dim=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )
    
    def forward(self, x):
        return self.net(x)

# ==============================================================
# REPLAY BUFFER
# ==============================================================

class ReplayBuffer:
    def __init__(self, cap=20000):
        self.buf = deque(maxlen=cap)
    
    def add(self, s, a, r, ns, d):
        self.buf.append((s, a, r, ns, d))
    
    def sample(self, batch):
        batch_data = random.sample(self.buf, batch)
        s, a, r, ns, d = zip(*batch_data)
        return (
            torch.from_numpy(np.asarray(s, dtype=np.float32)),
            torch.from_numpy(np.asarray(a, dtype=np.int64)),
            torch.from_numpy(np.asarray(r, dtype=np.float32)),
            torch.from_numpy(np.asarray(ns, dtype=np.float32)),
            torch.from_numpy(np.asarray(d, dtype=np.float32)),
        )
    
    def __len__(self):
        return len(self.buf)

# ==============================================================
# BASELINE POLICIES
# ==============================================================

class LocalPolicy:
    @staticmethod
    def decide(state, fog_backlog):
        return 0

class RandomPolicy:
    @staticmethod
    def decide(state, fog_backlog):
        return random.randint(0, 1)

class GreedyPolicy:
    @staticmethod
    def decide(state, fog_backlog):
        local_delay = state[0]
        path_delay = state[1]
        fog_delay = state[2]
        offload_delay = fog_backlog + path_delay + fog_delay
        return 1 if offload_delay < local_delay else 0

# ==============================================================
# TRAINING FUNCTION
# ==============================================================

def train_tetris_dqn(env, episodes=200):
    policy = OffloadDQN(4).to(DEVICE)
    target = deepcopy(policy)
    opt = optim.Adam(policy.parameters(), lr=1e-3)
    buf = ReplayBuffer()
    
    history = {
        "avg_delay": [], "hit_ratio": [], "avg_hops": [],
        "deadline_violation": [], "reward": [], "loss": [],
        "local_ratio": [], "offload_ratio": []
    }
    
    epsilon = 1.0
    
    print("\n" + "=" * 80)
    print("TRAINING TETRIS DQN")
    print("=" * 80)
    print(f"{'Ep':<6} {'Delay(ms)':<12} {'Hit Ratio':<12} {'Loss':<10} {'Reward':<12}")
    print("-" * 80)
    
    for ep in range(episodes):
        state = env.reset()
        ep_reward = 0
        ep_losses = []
        
        while True:
            s_t = torch.from_numpy(state).float().unsqueeze(0).to(DEVICE)
            
            if random.random() < epsilon:
                action = random.randint(0, 1)
            else:
                with torch.no_grad():
                    action = policy(s_t).argmax(1).item()
            
            next_state, reward, done, _ = env.step(action)
            buf.add(state, action, reward, next_state, done)
            state = next_state
            ep_reward += reward
            
            if len(buf) > 64:
                bs, ba, br, bns, bd = buf.sample(64)
                bs, ba, br, bns, bd = (
                    bs.to(DEVICE), ba.to(DEVICE), br.to(DEVICE),
                    bns.to(DEVICE), bd.to(DEVICE)
                )
                
                q = policy(bs).gather(1, ba.unsqueeze(1)).squeeze()
                with torch.no_grad():
                    nq = target(bns).max(1)[0]
                
                target_q = br + 0.99 * nq * (1 - bd)
                loss = ((q - target_q) ** 2).mean()
                ep_losses.append(loss.item())
                
                opt.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
                opt.step()
            
            if done:
                break
        
        epsilon = max(0.05, epsilon * 0.995)
        
        if ep % 10 == 0:
            target.load_state_dict(policy.state_dict())
        
        m = env.metrics()
        history["avg_delay"].append(m["avg_delay"])
        history["hit_ratio"].append(m["hit_ratio"])
        history["avg_hops"].append(m["avg_hops"])
        history["deadline_violation"].append(m["deadline_violation"])
        history["reward"].append(ep_reward)
        history["loss"].append(np.mean(ep_losses) if ep_losses else 0)
        history["local_ratio"].append(m["local_ratio"])
        history["offload_ratio"].append(m["offload_ratio"])
        
        if (ep + 1) % 20 == 0:
            print(f"{ep:<6} {m['avg_delay']*1000:<12.2f} {m['hit_ratio']:<12.3f} "
                  f"{np.mean(ep_losses) if ep_losses else 0:<10.4f} {ep_reward:<12.2f}")
    
    return policy, history

# ==============================================================
# EVALUATE BASELINES
# ==============================================================

def evaluate_baselines(trained_policy=None):
    """Evaluate all baselines"""
    path_estimator = PathEstimator()
    results = []
    
    policies = [
        ("Local", LocalPolicy()),
        ("Random", RandomPolicy()),
        ("Greedy", GreedyPolicy()),
    ]
    
    for name, policy in policies:
        env = TETRISOffloadEnv(path_estimator, num_tasks=200)
        hit_ratios = []
        avg_delays = []
        
        for ep in range(30):
            state = env.reset()
            done = False
            
            while not done:
                action = policy.decide(env.last_features, env.fog.backlog())
                state, _, done, _ = env.step(action)
            
            m = env.metrics()
            hit_ratios.append(m["hit_ratio"])
            avg_delays.append(m["avg_delay"] * 1000)
        
        results.append({
            "name": name,
            "hit_ratio": np.mean(hit_ratios),
            "hit_ratio_std": np.std(hit_ratios),
            "avg_delay": np.mean(avg_delays),
            "avg_delay_std": np.std(avg_delays)
        })
    
    # Evaluate TETRIS DQN if provided
    if trained_policy:
        env = TETRISOffloadEnv(path_estimator, num_tasks=200)
        hit_ratios = []
        avg_delays = []
        
        for ep in range(30):
            state = env.reset()
            done = False
            
            while not done:
                s_t = torch.from_numpy(state).float().unsqueeze(0).to(DEVICE)
                with torch.no_grad():
                    action = trained_policy(s_t).argmax(1).item()
                state, _, done, _ = env.step(action)
            
            m = env.metrics()
            hit_ratios.append(m["hit_ratio"])
            avg_delays.append(m["avg_delay"] * 1000)
        
        results.append({
            "name": "TETRIS",
            "hit_ratio": np.mean(hit_ratios),
            "hit_ratio_std": np.std(hit_ratios),
            "avg_delay": np.mean(avg_delays),
            "avg_delay_std": np.std(avg_delays)
        })
    
    return results

# ==============================================================
# SAVE RESULTS
# ==============================================================

def save_results(history, baseline_results, model, save_dir="tetris_results"):
    """Save all training results"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = os.path.join(save_dir, timestamp)
    os.makedirs(save_path, exist_ok=True)
    
    # Save training history
    np.savez(
        os.path.join(save_path, "training_history.npz"),
        avg_delay=np.array(history["avg_delay"]),
        hit_ratio=np.array(history["hit_ratio"]),
        avg_hops=np.array(history["avg_hops"]),
        deadline_violation=np.array(history["deadline_violation"]),
        reward=np.array(history["reward"]),
        loss=np.array(history["loss"]),
        local_ratio=np.array(history["local_ratio"]),
        offload_ratio=np.array(history["offload_ratio"])
    )
    
    # Save baseline results
    with open(os.path.join(save_path, "baseline_results.json"), 'w') as f:
        json.dump(baseline_results, f, indent=2)
    
    # Save model
    torch.save(model.state_dict(), os.path.join(save_path, "tetris_dqn.pth"))
    
    # Save final metrics
    with open(os.path.join(save_path, "final_metrics.txt"), 'w') as f:
        f.write("TETRIS FINAL METRICS\n")
        f.write("=" * 40 + "\n")
        f.write(f"Final Hit Ratio: {history['hit_ratio'][-1]*100:.2f}%\n")
        f.write(f"Final Avg Delay: {history['avg_delay'][-1]*1000:.2f} ms\n")
        f.write(f"Final Deadline Violation: {history['deadline_violation'][-1]*100:.2f}%\n")
        f.write(f"Final Reward: {history['reward'][-1]:.2f}\n")
    
    print(f"\n✅ Results saved to: {save_path}")
    return save_path

# ==============================================================
# MAIN
# ==============================================================

if __name__ == "__main__":
    print("=" * 80)
    print("TETRIS: Joint Task Routing and Offloading")
    print("=" * 80)
    print(f"Device: {DEVICE}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
    print("=" * 80)
    
    # Initialize
    print("\nInitializing Path Estimator...")
    path_estimator = PathEstimator()
    
    print("Initializing Offload Environment...")
    env = TETRISOffloadEnv(path_estimator, num_tasks=200)
    
    # Train
    print("\nStarting Training...")
    trained_policy, history = train_tetris_dqn(env, episodes=200)
    
    # Evaluate baselines
    print("\nEvaluating Baselines...")
    baseline_results = evaluate_baselines(trained_policy)
    
    # Print results
    print("\n" + "=" * 80)
    print("BASELINE COMPARISON")
    print("=" * 80)
    print(f"{'Policy':<12} {'Hit Ratio (%)':<18} {'Avg Delay (ms)':<18}")
    print("-" * 80)
    for r in baseline_results:
        print(f"{r['name']:<12} {r['hit_ratio']*100:<18.1f} {r['avg_delay']:<18.2f}")
    
    # Save results
    save_path = save_results(history, baseline_results, trained_policy)
    
    print("\n" + "=" * 80)
    print("TRAINING COMPLETE!")
    print("=" * 80)
    print(f"Final Hit Ratio: {history['hit_ratio'][-1]*100:.1f}%")
    print(f"Final Avg Delay: {history['avg_delay'][-1]*1000:.2f} ms")

Using device: cuda
TETRIS: Joint Task Routing and Offloading
Device: cuda
GPU: NVIDIA GeForce RTX 3090

Initializing Path Estimator...
Initializing Offload Environment...

Starting Training...

TRAINING TETRIS DQN
Ep     Delay(ms)    Hit Ratio    Loss       Reward      
--------------------------------------------------------------------------------
19     10.31        0.650        128.1156   -3706.10    
39     9.84         0.685        70.7129    -3346.89    
59     8.38         0.820        40.5126    -1967.57    
79     7.62         0.895        37.2144    -1202.37    
99     8.10         0.875        36.0630    -1412.04    
119    7.84         0.880        36.1886    -1356.73    
139    8.29         0.850        32.3740    -1665.72    
159    7.56         0.890        32.0088    -1251.12    
179    7.56         0.905        31.8666    -1101.15    
199    7.64         0.910        24.5119    -1052.77    

Evaluating Baselines...

BASELINE COMPARISON
Policy       Hit Ratio (%)      

In [43]:
import numpy as np
import torch
import json
import os
from datetime import datetime

def save_training_results(history, baseline_results, model, save_dir="tetris_results"):
    """Save all training results after completion"""
    
    # Create timestamped directory
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    save_path = os.path.join(save_dir, timestamp)
    os.makedirs(save_path, exist_ok=True)
    
    # 1. Save training history as NPZ
    np.savez(
        os.path.join(save_path, "training_history.npz"),
        avg_delay=np.array(history["avg_delay"]),
        hit_ratio=np.array(history["hit_ratio"]),
        avg_hops=np.array(history["avg_hops"]),
        deadline_violation=np.array(history["deadline_violation"]),
        reward=np.array(history["reward"]),
        loss=np.array(history["loss"]),
        local_ratio=np.array(history["local_ratio"]),
        offload_ratio=np.array(history["offload_ratio"])
    )
    
    # 2. Save baseline results
    with open(os.path.join(save_path, "baseline_results.json"), 'w') as f:
        json.dump(baseline_results, f, indent=2)
    
    # 3. Save trained model
    torch.save(model.state_dict(), os.path.join(save_path, "tetris_dqn.pth"))
    
    # 4. Save final metrics as text
    with open(os.path.join(save_path, "final_metrics.txt"), 'w') as f:
        f.write("TETRIS FINAL METRICS\n")
        f.write("=" * 40 + "\n")
        f.write(f"Final Hit Ratio: {history['hit_ratio'][-1]*100:.2f}%\n")
        f.write(f"Final Avg Delay: {history['avg_delay'][-1]*1000:.2f} ms\n")
        f.write(f"Final Deadline Violation: {history['deadline_violation'][-1]*100:.2f}%\n")
        f.write(f"Final Reward: {history['reward'][-1]:.2f}\n")
        f.write(f"Final Loss: {history['loss'][-1]:.4f}\n")
    
    print(f"\n✅ Results saved to: {save_path}")
    print(f"   - training_history.npz")
    print(f"   - baseline_results.json")
    print(f"   - tetris_dqn.pth")
    print(f"   - final_metrics.txt")
    
    return save_path